# Chromadb 구축

#### 환경 설정

In [2]:
# 라이브러리와 한국어 임베딩 모델
import numpy as np
import pandas as pd
import json
from sklearn.metrics.pairwise import cosine_similarity
import chromadb
from typing import List
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings


# 한국어 임베딩 모델 — 문장 한 개를 768차원 벡터로 변환
emb_model = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={"device": "cpu"},  # GPU 사용 시 'cuda'
    encode_kwargs={"normalize_embeddings": True},
)
print('임베딩 모델 준비 완료 — 벡터 차원:', emb_model.__getstate__)

d:\encore\mle-01-p1-team3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6644.75it/s]


임베딩 모델 준비 완료 — 벡터 차원: <bound method BaseModel.__getstate__ of HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': True}, query_encode_kwargs={}, multi_process=False, show_progress=False)>


#### 데이터 가져오기

#### Langchain 활용 chromadb 구축

#### 청킹 데이터 3개 통합

In [3]:
from pathlib import Path


# 프로젝트 구조에 맞는 데이터 폴더
DATA_DIR = "../data/RAG/"

def load_documents(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    documents = [
        Document(page_content=doc["page_content"], metadata=doc["metadata"])
        for doc in data
    ]

    return documents

guide_documents = load_documents(DATA_DIR + "maple_guides_documents_chunked.json")
jobs_documents = load_documents(DATA_DIR + "maple_jobs_documents.json")
items_documents = load_documents(DATA_DIR + "maple_items_documents.json")

for doc in guide_documents:
    doc.metadata['source'] = "guide"
for doc in jobs_documents:
    doc.metadata['source'] = "jobs"
for doc in items_documents:
    doc.metadata['source'] = "items"

all_documents = guide_documents + jobs_documents + items_documents

In [4]:

def build_maplestory_chromadb(
    chunked_documents: List[Document], 
    persist_directory: str = "../chroma_db",
    collection_name: str = "maplestory_guides"
) -> Chroma:
    """
    메이플스토리 청크 문서를 ChromaDB에 스키마를 맞춰 저장합니다.
    """
    # 문서 고유 ID 목록 생성 (chunk_{chunk_index})
    try : 
        ids = []
        for idx, doc in enumerate(chunked_documents):
            chunk_index = doc.metadata.get('chunk_index', idx)
            ids.append(f"chunk_{chunk_index}")
            
        # ChromaDB 생성 및 저장 (Cosine Distance 기준)
        vectorstore = Chroma.from_documents(
            documents=chunked_documents,
            embedding=emb_model,
            ids=ids,
            collection_name=collection_name,
            persist_directory=persist_directory,
            collection_metadata={"hnsw:space": "cosine"}  # 유사도 계산 방식 설정 (cosine, l2, ip)
        )
        
        print(f"ChromaDB에 {len(chunked_documents)}개의 청크가 성공적으로 저장되었습니다.")
        return vectorstore
    except Exception as e :
        print(f'오류가 발생했습니다. : {e}')

build_maplestory_chromadb(all_documents)

ChromaDB에 3694개의 청크가 성공적으로 저장되었습니다.
